# Banking intent routing: inspecting the experiment

This notebook walks through the frozen benchmark and its failure cases. Training lives in `intentlab/train.py`. MiniLM is pretrained and frozen; the classifier, temperature and review threshold are fitted using separate development partitions.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

In [2]:
from intentlab.data import load_data

train, test, splits, removed = load_data()
print("Removed training duplicates/overlaps:", removed)
print("Intent categories:", train.category.nunique())
pd.Series({**{k: len(v) for k, v in splits.items()}, "test": len(test)}, name="rows").to_frame()

Removed training duplicates/overlaps: 11
Intent categories: 77


,rows
train,6994
calibration,1499
validation,1499
test,3080


## Split integrity

The official test split stays untouched. Exact normalized text overlap is checked, though semantic near-duplicates can remain.

In [3]:
from itertools import combinations

for left, right in combinations(splits, 2):
    assert set(train.loc[splits[left], "normalized"]).isdisjoint(
        train.loc[splits[right], "normalized"]
    )
assert set(train.normalized).isdisjoint(test.normalized)
print("No exact normalized-text leakage across partitions.")

No exact normalized-text leakage across partitions.


In [4]:
metrics = json.loads(Path("reports/metrics.json").read_text())
print("Selected:", metrics["selected_model"])
pd.DataFrame(metrics["test"]).T[["accuracy", "macro_f1", "log_loss", "ece_10_bins"]].round(4)

Selected: minilm_logistic_c16


,accuracy,macro_f1,log_loss,ece_10_bins
tfidf_logistic,0.875974,0.875838,0.452814,0.014779
minilm_logistic_c4,0.920779,0.920671,0.285242,0.011409
minilm_logistic_c16,0.930844,0.930761,0.264554,0.012948
minilm_logistic_c64,0.927922,0.927815,0.271986,0.014335


## Accuracy depends on coverage

The accepted-request denominator is smaller than the full test set. Deferred requests are not silently counted as correct. The threshold comes from validation; the independent test result evaluates that frozen policy.

In [5]:
pd.DataFrame([metrics["routing_test"], metrics["typo_routing"]], index=["clean", "typo stress"])[
    ["threshold", "accepted", "total", "coverage", "accuracy"]
].round(4)

,threshold,accepted,total,coverage,accuracy
clean,0.68,2792,3080,0.9065,0.9692
typo stress,0.68,2226,3080,0.7227,0.9349


In [6]:
pd.read_csv("reports/confusions.csv").head(10)

,count,actual,predicted
0,4,pending_transfer,transfer_timing
1,4,card_payment_not_recognised,compromised_card
2,4,card_delivery_estimate,card_arrival
3,3,pending_transfer,transfer_not_received_by_recipient
4,3,get_disposable_virtual_card,getting_virtual_card
5,3,fiat_currency_support,exchange_via_app
6,3,declined_transfer,declined_card_payment
7,3,card_payment_not_recognised,direct_debit_payment_not_recognised
8,3,card_arrival,card_delivery_estimate
9,3,balance_not_updated_after_bank_transfer,transfer_timing


## High confidence can still be wrong

Every probe below was intended to need review. Inspect the automatically accepted failures instead of relying only on average test accuracy. These 24 authored probes are diagnostic, not representative traffic.

In [7]:
probes = pd.read_json("reports/review_probe_results.json")
probes.loc[~probes.reviewed, ["text", "kind", "predicted", "confidence"]]

,text,kind,predicted,confidence
6,My dog is refusing to eat.,out_of_scope,card_not_working,0.862773
10,How do I apply for a mortgage?,unsupported_banking,failed_transfer,0.793451
22,I want to change something.,ambiguous,edit_personal_details,0.985845
23,This amount looks wrong.,ambiguous,wrong_amount_of_cash_received,0.996046


## Next experiment

Use separate noisy/out-of-domain development data for a rejection model and a fresh holdout for evaluation. Do not tune the policy using the errors above and call the original test an untouched evaluation. No staffing or dollar savings have been measured.